In [1]:
import os
import pyarrow as pa     # импорт необходим, чтобы не крашился ноутбук на Windows
import numpy as np
import polars as pl
import torch

import src.dataset as dts
from src.modeling.vae import run_vae_experiment
from src.modeling.train_vae import build_user_history_dataset, train_vae
from src.config import VAE_DIR

import shutil
import tempfile
from pathlib import Path
from scipy.sparse import load_npz

from src.modeling.vae import MultiVAERecommender
from src.dataset import ndcg_at_k, precision_recall_at_k
print("torch:", torch.__version__)

2026-06-03 23:33:05.224 | INFO     | src.config:<module>:12 - PROJ_ROOT path is: D:\HSE_repos\dreamteam-recsys


torch: 2.12.0+cu130


In [2]:
EVENTS_DIR = "../data/raw/dataset/small/marketplace/events"

events_df = pl.scan_parquet(f"{EVENTS_DIR}/*.pq", include_file_paths="path").sort("day")

In [3]:
# Inverse-frequency target — тот же, что в SVD/iALS-ноутбуках для конформности.
actions_count = (
    events_df.group_by("action_type")
    .agg(pl.len())
    .collect(engine="streaming")
    .with_columns(
        (pl.col("len").sum() / pl.col("len")).alias("target"),
        (pl.col("len").sum() / pl.col("len")).log1p().alias("log_target"),
        (pl.col("len").sum() / pl.col("len")).sqrt().alias("sqrt_target"),
    )
    .drop("len")
)
events_df = events_df.join(actions_count.lazy(), on="action_type")
actions_count

action_type,target,log_target,sqrt_target
str,f64,f64,f64
"""view""",1.034082,0.710044,1.016898
"""like""",3121.341812,8.046339,55.86897
"""click""",34.582149,3.571844,5.880659
"""clickout""",268.714913,5.597366,16.392526


In [4]:
train, test = dts.global_temporal_split(events_df, 0.1)
print(f"Train schema: {train.collect_schema()}")
print(f"Test schema:  {test.collect_schema()}")

Train schema: Schema({'timestamp': Duration(time_unit='us'), 'user_id': UInt64, 'item_id': String, 'subdomain': String, 'action_type': String, 'os': String, 'day': Int32, 'path': String, 'target': Float64, 'log_target': Float64, 'sqrt_target': Float64})
Test schema:  Schema({'timestamp': Duration(time_unit='us'), 'user_id': UInt64, 'item_id': String, 'subdomain': String, 'action_type': String, 'os': String, 'day': Int32, 'path': String, 'target': Float64, 'log_target': Float64, 'sqrt_target': Float64})


In [5]:
best_params = {
    "target_col": "log_target",
    "latent_dim": 64,
    "hidden_dim": 800,
    "dropout": 0.7,
    "beta": 0.215,
    "epochs": 25,
    "binary": False,
    "batch_size": 360
}

In [6]:
def build_history_from_events(
    train_lf: pl.LazyFrame,
    target_col: str,
    output_path: Path,
    binary: bool = True,
    min_interactions: int = 3,
    min_day: int | None = None,
) -> tuple[dict[str, int], dict[int, int]]:
    """Строит sparse user-history parquet из LazyFrame событий.

    Выходная схема: (user_idx: u32, user_id: u64, item_idx: list[u32], weight: list[f32]).
    train_lf должен уже быть отфильтрован по нужным item_id.

    Parameters
    ----------
    min_day : int or None
        Если задан, используются только события с ``day >= min_day``.
        Позволяет ограничить историю конкретным временным окном,
        аналогично снимку ``build_user_history_dataset(day=DAY)``.
    """
    if min_day is not None:
        train_lf = train_lf.filter(pl.col("day") >= min_day)

    df = (
        train_lf
        .select("user_id", "item_id", target_col)
        .collect(engine="streaming")
        .group_by("user_id", "item_id")
        .agg(pl.col(target_col).sum())
    )

    valid_users = (
        df.group_by("user_id").agg(pl.len().alias("cnt"))
        .filter(pl.col("cnt") >= min_interactions).select("user_id")
    )
    valid_items = (
        df.group_by("item_id").agg(pl.len().alias("cnt"))
        .filter(pl.col("cnt") >= min_interactions).select("item_id")
    )
    df = df.join(valid_users, on="user_id").join(valid_items, on="item_id")

    item_ids_sorted = df.select("item_id").unique().sort("item_id")["item_id"].to_list()
    item_to_idx: dict[str, int] = {iid: idx for idx, iid in enumerate(item_ids_sorted)}
    items_meta = pl.DataFrame(
        {"item_id": item_ids_sorted, "item_idx": list(range(len(item_ids_sorted)))},
        schema={"item_id": pl.String, "item_idx": pl.UInt32},
    )

    if binary:
        df = df.with_columns(weight=pl.lit(1.0, dtype=pl.Float32))
    else:
        df = df.with_columns(weight=pl.col(target_col).cast(pl.Float32))

    df = df.join(items_meta, on="item_id")

    user_df = (
        df.group_by("user_id")
        .agg(pl.col("item_idx"), pl.col("weight"))
        .sort("user_id")
        .with_row_index("user_idx")
        .with_columns(pl.col("user_idx").cast(pl.UInt32))
    )
    user_to_idx: dict[int, int] = dict(
        zip(user_df["user_id"].to_list(), user_df["user_idx"].to_list())
    )

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    user_df.select("user_idx", "user_id", "item_idx", "weight").write_parquet(output_path)
    return item_to_idx, user_to_idx


def evaluate_ndcg(
    rec: MultiVAERecommender,
    test_lf: pl.LazyFrame,
    target_col: str,
    top_k: int = 15,
) -> dict:
    """Считает NDCG/Precision/Recall на тестовой выборке."""
    test_df = test_lf.select("user_id", "item_id", target_col).collect(engine="streaming")
    known_users = set(rec.user_to_idx.keys())
    test_filtered = test_df.filter(pl.col("user_id").is_in(known_users))

    user_test_truth = test_filtered.group_by("user_id").agg(
        pl.col("item_id").alias("true_items"),
        pl.col(target_col).alias("relevancy"),
    )
    test_user_ids = user_test_truth["user_id"].to_numpy()
    test_user_idxs = np.array([rec.user_to_idx[uid] for uid in test_user_ids])

    BATCH = 1024
    item_idx_chunks, score_chunks = [], []
    for start in range(0, len(test_user_idxs), BATCH):
        chunk = test_user_idxs[start : start + BATCH]
        ids, scores = rec.recommend_batch(chunk, top_k=top_k)
        item_idx_chunks.append(ids)
        score_chunks.append(scores)

    item_idx_matrix = np.concatenate(item_idx_chunks, axis=0)
    score_matrix    = np.concatenate(score_chunks,    axis=0)

    predicted_items_list  = [[rec.idx_to_item[i] for i in row] for row in item_idx_matrix]
    predicted_scores_list = [row.tolist() for row in score_matrix]

    prediction_df  = pl.DataFrame({
        "user_id":          test_user_ids,
        "predicted_items":  predicted_items_list,
        "predicted_scores": predicted_scores_list,
    })
    evaluation_df = user_test_truth.join(prediction_df, on="user_id")

    ndcg_results = ndcg_at_k(
        evaluation_df,
        relevancy_col="relevancy",
        true_items_col="true_items",
        predicted_items_col="predicted_items",
        predicted_score_col="predicted_scores",
        top_k=top_k,
    )
    mean_ndcg = float(ndcg_results["ndcg"].mean())
    precision, recall = precision_recall_at_k(
        evaluation_df,
        predicted_items_col="predicted_items",
        true_items_col="true_items",
        top_k=top_k,
    )
    return {"ndcg": mean_ndcg, "precision": precision, "recall": recall}

In [7]:
DAY = 1276
SEED = 42
TOP_K= 15

In [8]:
from dotenv import load_dotenv
import mlflow
import mlflow.exceptions

load_dotenv("../.env") 

# Переопределяю адрес, так как у меня заняты порты на локальной машине
# os.environ["MLFLOW_S3_ENDPOINT_URL"] = "http://localhost:19000"
# ====== S3 / MinIO конфиг ======
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("MINIO_ROOT_USER")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("MINIO_ROOT_PASSWORD")

# важно: чтобы MLflow понимал, что это MinIO, а не AWS
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "http://localhost:9000"

mlflow.set_tracking_uri("http://localhost:5050")
experiment_name = "MultVAE"

try:
    exp_id = mlflow.create_experiment(
        name=experiment_name,
        artifact_location=f"s3://{os.getenv("DEFAULT_BUCKET_NAME")}"
    )
except mlflow.exceptions.MlflowException:
    exp_id = mlflow.get_experiment_by_name(experiment_name).experiment_id

mlflow.set_experiment(experiment_name)

print(f"Подключено к MLflow. Эксперимент: {experiment_name}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

Подключено к MLflow. Эксперимент: MultVAE
Tracking URI: http://localhost:5050


In [9]:
ITEMS_TO_TAKE = 20_000

all_items_ids = (
    train.group_by("item_id").agg(count=pl.len()).sort("count", descending=True).head(ITEMS_TO_TAKE).collect(engine="streaming")["item_id"].to_list()
)

In [10]:
import json


with mlflow.start_run(run_name="multvae_full_data", tags={"stage": "PRD"}) as run:

    # Фильтруем train по тем же 40k item-ам и тому же 30-дневному окну [DAY-29, DAY],
    # что и в Optuna — чтобы финальная модель была сравнима с гиперопт-триалами
    train_full = train.filter(pl.col("item_id").is_in(all_items_ids))

    print("\nОбучение на полном датасете (OOM)...")
    _tmp_dir = Path(tempfile.mkdtemp(prefix="vae_final_"))
    try:
        _history_path = _tmp_dir / "history.pq"
        _artifacts_dir = _tmp_dir / "artifacts"

        _item_to_idx, _user_to_idx = build_history_from_events(
            train_full,
            target_col=best_params["target_col"],
            output_path=_history_path,
            binary=best_params["binary"],
            min_interactions=5,
            min_day=DAY - 29,
        )
        print(f"  {len(_user_to_idx):,} пользователей x {len(_item_to_idx):,} items")

        mlflow.log_params(best_params)
        train_vae(
            history_path=_history_path,
            item_to_idx=_item_to_idx,
            user_to_idx=_user_to_idx,
            artifacts_dir=_artifacts_dir,
            encoder_dims=[best_params["hidden_dim"], best_params["latent_dim"]],
            dropout=best_params["dropout"],
            beta=best_params["beta"],
            epochs=best_params["epochs"],
            batch_size=best_params["batch_size"],
            total_anneal_steps=200_000,
            use_lr_scheduler=True,
            num_workers=12,
            seed=SEED,
        )

        _rec = MultiVAERecommender()
        _rec.load(_artifacts_dir / "model.pt")
        _rec.set_user_items(load_npz(_artifacts_dir / "user_items.npz"))

        _metrics_train = evaluate_ndcg(
            _rec, train, target_col=best_params["target_col"], top_k=TOP_K
        )
        _metrics_train = {f"train_{k}": v for k, v in _metrics_train.items()}
        _metrics_test = evaluate_ndcg(_rec, test, target_col=best_params["target_col"], top_k=TOP_K)
        _metrics_test = {f"test_{k}": v for k, v in _metrics_test.items()}
        
        mlflow.log_metrics({**_metrics_train, **_metrics_test})

        _rec.save(_artifacts_dir / "model")
        
        with open(_artifacts_dir / "items_ids.json", "w") as f:
            json.dump(all_items_ids, f)
        mlflow.log_artifact(str(_artifacts_dir / "model"))
        mlflow.log_artifact(str(_artifacts_dir / "items_ids.json"))
        mlflow.log_artifact(str(_artifacts_dir / "user_items.npz"))
        # mlflow.log_artifact(str(_artifacts_dir))
        print("Run ID:", run.info.run_id)
    finally:
        shutil.rmtree(_tmp_dir, ignore_errors=True)


Обучение на полном датасете (OOM)...
  628,315 пользователей x 14,522 items
2026-06-03 23:33:19.600 | INFO     | src.modeling.vae:build_model:385 - MultiVAE собрана: num_items=14522, dims=[800, 64], device=cuda


Generating train split: 0 examples [00:00, ? examples/s]

2026-06-03 23:33:20.081 | INFO     | src.modeling.train_vae:train_vae:370 - Стартую обучение: 628315 пользователей, 14522 items, batches per epoch ≈ 1746


Эпохи:   4%|▍         | 1/25 [00:40<16:08, 40.36s/epoch, beta=0.002, kld=196.997, loss=156.929, lr=9.96e-04, nll=156.728]

2026-06-03 23:34:27.472 | INFO     | src.modeling.train_vae:train_vae:468 - Epoch 01: loss=156.9289 nll=156.7283 kld=196.9974 beta=0.0019 lr=9.96e-04 time=40.4s


...

2026-06-03 23:48:53.240 | INFO     | src.modeling.train_vae:train_vae:468 - Epoch 23: loss=139.3565 nll=136.1432 kld=76.0936 beta=0.0432 lr=2.56e-05 time=41.1s


Эпохи:  96%|█████████▌| 24/25 [15:46<00:40, 40.41s/epoch, beta=0.045, kld=74.873, loss=139.401, lr=1.39e-05, nll=136.099]

2026-06-03 23:49:34.091 | INFO     | src.modeling.train_vae:train_vae:468 - Epoch 24: loss=139.4008 nll=136.0985 kld=74.8729 beta=0.0450 lr=1.39e-05 time=40.8s


Эпохи: 100%|██████████| 25/25 [16:27<00:00, 39.50s/epoch, beta=0.047, kld=73.629, loss=139.524, lr=1.00e-05, nll=136.138]


2026-06-03 23:50:14.638 | INFO     | src.modeling.train_vae:train_vae:468 - Epoch 25: loss=139.5237 nll=136.1380 kld=73.6288 beta=0.0469 lr=1.00e-05 time=40.5s
2026-06-03 23:50:15.098 | INFO     | src.modeling.vae:save:477 - MultiVAE сохранена в C:\Users\Sasha\AppData\Local\Temp\vae_final_f5is07j5\artifacts\model.pt
2026-06-03 23:50:22.760 | INFO     | src.modeling.train_vae:train_vae:490 - user_items.npz сохранена в C:\Users\Sasha\AppData\Local\Temp\vae_final_f5is07j5\artifacts
2026-06-03 23:50:23.744 | INFO     | src.modeling.vae:build_model:385 - MultiVAE собрана: num_items=14522, dims=[800, 64], device=cuda
2026-06-03 23:50:23.762 | INFO     | src.modeling.vae:load:497 - MultiVAE загружена из C:\Users\Sasha\AppData\Local\Temp\vae_final_f5is07j5\artifacts\model.pt
2026-06-03 23:54:55.293 | INFO     | src.modeling.vae:save:477 - MultiVAE сохранена в C:\Users\Sasha\AppData\Local\Temp\vae_final_f5is07j5\artifacts\model
Run ID: 57a7576a36394e87829afd763ca77a02
🏃 View run multvae_full_da

## Проверяем устойчивость

In [ ]:
import numpy as np
from mlflow.artifacts import download_artifacts
from scipy.sparse import load_npz

TOP_K = 10
N_USERS = 1000
REMOVE_ITEMS = 1


def jaccard(a, b):
    a = set(a)
    b = set(b)

    if not (a or b):
        return 1.0

    return len(a & b) / len(a | b)


def stability(a, b):
    a = set(a)
    b = set(b)

    return len(a & b) / len(a)


def load_model_from_mlflow(run_id):

    model_file = download_artifacts(
        artifact_uri=f"runs:/{run_id}/model"
    )

    rec = MultiVAERecommender()
    rec.load(model_file)

    return rec

def load_npz_from_mlflow(run_id):
    user_items_path = download_artifacts(
        artifact_uri=f"runs:/{run_id}/user_items.npz"
    )

    return load_npz(user_items_path)


def robustness_test(
    rec,
    user_items,
    top_k=10,
    n_users=1000,
    remove_items=1,
):

    rec.set_user_items(user_items)

    all_users = np.arange(user_items.shape[0])

    if len(all_users) > n_users:
        all_users = np.random.choice(
            all_users,
            size=n_users,
            replace=False,
        )

    stabilities = []
    jaccards = []
    changed_positions = []

    for user_idx in all_users:

        history = user_items[user_idx].copy()

        interacted = history.indices

        if len(interacted) <= remove_items:
            continue

        original_items, _ = rec.recommend_batch(
            np.array([user_idx]),
            top_k=top_k,
        )

        original_rec = original_items[0]

        perturbed_history = history.copy()

        removed = np.random.choice(
            interacted,
            size=remove_items,
            replace=False,
        )

        for item in removed:
            perturbed_history[0, item] = 0

        original_user_history = rec.user_items[user_idx]
        rec.user_items[user_idx] = perturbed_history

        perturbed_items, _ = rec.recommend_batch(
            np.array([user_idx]),
            top_k=top_k,
        )

        perturbed_rec = perturbed_items[0]

        rec.user_items[user_idx] = original_user_history

        stabilities.append(
            stability(original_rec, perturbed_rec)
        )

        jaccards.append(
            jaccard(original_rec, perturbed_rec)
        )

        changed_positions.append(
            np.sum(original_rec != perturbed_rec)
        )

    return {
        "robustness_topk_stability_mean": float(np.mean(stabilities)),
        "robustness_topk_stability_std": float(np.std(stabilities)),
        "robustness_jaccard_mean": float(np.mean(jaccards)),
        "robustness_jaccard_std": float(np.std(jaccards)),
        "robustness_changed_items_mean": float(np.mean(changed_positions)),
        "robustness_users_count": int(len(stabilities)),
    }

In [13]:
RUN_ID = "57a7576a36394e87829afd763ca77a02"


rec = load_model_from_mlflow(RUN_ID)

user_items = load_npz_from_mlflow(RUN_ID)


for top_k in [10, 20, 30]:
    for remove_items in [1, 2, 3]:

        with mlflow.start_run(run_name="multvae_robustness_test"):

            mlflow.log_param("source_run_id", RUN_ID)
            mlflow.log_param("top_k", top_k)
            mlflow.log_param("remove_items", remove_items)

            metrics = robustness_test(
                rec,
                user_items,
                top_k=top_k,
                n_users=1000,
                remove_items=remove_items,
            )

            mlflow.log_metrics(metrics)

2026-06-03 23:57:09.980 | INFO     | src.modeling.vae:build_model:385 - MultiVAE собрана: num_items=14522, dims=[800, 64], device=cuda
2026-06-03 23:57:09.991 | INFO     | src.modeling.vae:load:497 - MultiVAE загружена из C:\Users\Sasha\AppData\Local\Temp\tmpk3ot8tts\model


🏃 View run multvae_robustness_test at: http://localhost:5050/#/experiments/2/runs/0bc2e131dfc7468dad3b30da868c2381
🧪 View experiment at: http://localhost:5050/#/experiments/2
🏃 View run multvae_robustness_test at: http://localhost:5050/#/experiments/2/runs/962bf63d0e454bd0986b4b4812bc65bf
🧪 View experiment at: http://localhost:5050/#/experiments/2
🏃 View run multvae_robustness_test at: http://localhost:5050/#/experiments/2/runs/be71d2e39cfc40d896112f854d1bf6bc
🧪 View experiment at: http://localhost:5050/#/experiments/2
🏃 View run multvae_robustness_test at: http://localhost:5050/#/experiments/2/runs/7ac6691ef88d4507bdbadc02ae5d2e37
🧪 View experiment at: http://localhost:5050/#/experiments/2
🏃 View run multvae_robustness_test at: http://localhost:5050/#/experiments/2/runs/96e941a1136f4c818c450465a77195ff
🧪 View experiment at: http://localhost:5050/#/experiments/2
🏃 View run multvae_robustness_test at: http://localhost:5050/#/experiments/2/runs/295ac58eeb2141afa4f134b51c573cb3
🧪 View exp

## Загружаем продовую модель

In [ ]:

exp = mlflow.get_experiment_by_name("MultVAE")

runs = mlflow.search_runs(
    experiment_ids=[exp.experiment_id],
    filter_string="tags.stage = 'PRD'",
    order_by=["start_time DESC"],
    max_results=1
)

latest_run_id = runs.iloc[0]["run_id"]

57a7576a36394e87829afd763ca77a02


In [16]:
model = load_model_from_mlflow(latest_run_id)

2026-06-04 00:04:48.199 | INFO     | src.modeling.vae:build_model:385 - MultiVAE собрана: num_items=14522, dims=[800, 64], device=cuda
2026-06-04 00:04:48.215 | INFO     | src.modeling.vae:load:497 - MultiVAE загружена из C:\Users\Sasha\AppData\Local\Temp\tmpafzwjx04\model


In [ ]:
model.set_user_items(load_npz_from_mlflow(latest_run_id))

In [51]:
test_users = (
    test.select("user_id")
    .filter(pl.col("user_id").is_in(model.user_to_idx.keys()))
    .head(20)
    .collect()["user_id"]
    .to_list()
)

In [61]:
test_users

[50145974,
 14042715,
 69416985,
 8116339,
 44852811,
 77485130,
 14443163,
 59670886,
 80598643,
 84508206,
 54083133,
 26229418,
 88831493,
 20428132,
 46443338,
 60195175,
 4463183,
 36260945,
 4169312,
 85219578]

In [52]:
test_users_idx = np.array(
    list(
        x
        for x in (
            map(
                model.user_to_idx.get,
                test_users,
            )
        )
        if x is not None
    )
)

In [56]:
recs = model.recommend_batch(test_users_idx, top_k=10)[0]

In [58]:
recs.shape

(20, 10)

In [59]:
recommendations = np.vectorize(model.idx_to_item.get)(recs)
recommendations_df = pl.DataFrame({"user_id": test_users,"recommendations": list(x for x in recommendations)})

In [60]:
recommendations_df

user_id,recommendations
i64,"array[str, 10]"
50145974,"[""nfmcg_20564139"", ""nfmcg_22178359"", … ""nfmcg_23097146""]"
14042715,"[""nfmcg_18106422"", ""nfmcg_14380950"", … ""nfmcg_7763862""]"
69416985,"[""nfmcg_7537528"", ""nfmcg_8961427"", … ""nfmcg_16735605""]"
8116339,"[""nfmcg_716576"", ""nfmcg_12977198"", … ""nfmcg_1757172""]"
44852811,"[""nfmcg_23226394"", ""nfmcg_7997336"", … ""nfmcg_2845426""]"
…,…
60195175,"[""nfmcg_2501643"", ""nfmcg_18954296"", … ""nfmcg_8961427""]"
4463183,"[""nfmcg_27366316"", ""nfmcg_716576"", … ""nfmcg_7997336""]"
36260945,"[""nfmcg_7763862"", ""nfmcg_22204691"", … ""nfmcg_15720406""]"
